<a href="https://colab.research.google.com/github/chrishg23-jpg/HES-benchmark/blob/main/HES20GR001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Act GR-1 (Stable Patch): Spectral Poisson solve and weak-field Einstein check
# Periodic box, double precision, dimensionless normalization

import numpy as np

# --- Precision & grid ---
np.random.seed(0)
N = 64
L = 1.0
dx = L / N
dtype = np.float64

# --- Dimensionless normalization ---
# We absorb 4*pi*G into rho scaling to avoid huge coefficients.
rho0 = 1.0

# --- Coordinates ---
x = np.linspace(0, L, N, endpoint=False, dtype=dtype)
X, Y, Z = np.meshgrid(x, x, x, indexing='ij')

# --- Smooth static density (periodic) ---
sigma = 0.12  # wider blob to reduce high-k content
rho = rho0 * np.exp(-((X-0.5)**2 + (Y-0.5)**2 + (Z-0.5)**2) / (sigma**2)).astype(dtype)

# --- Normalize: absorb 4*pi into rho so Poisson is ∇^2 Phi = rho ---
rho /= (4.0 * np.pi)

# --- Spectral Poisson solver (periodic BCs) ---
# Poisson: k^2 Phi_k = -rho_k  ->  Phi_k = -rho_k / k^2, with k=0 mode handled separately
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
kz = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
k2 = (KX**2 + KY**2 + KZ**2).astype(dtype)

rho_k = np.fft.fftn(rho)
Phi_k = np.zeros_like(rho_k, dtype=np.complex128)

# Avoid division by zero at k=0: set mean(phi) = 0 (gauge choice in periodic box)
mask = k2 != 0.0
Phi_k[mask] = -rho_k[mask] / k2[mask]
Phi_k[~mask] = 0.0

Phi = np.real(np.fft.ifftn(Phi_k)).astype(dtype)

# --- Discrete Laplacian (central differences, periodic) ---
def laplacian(phi, dx):
    return (
        (np.roll(phi, 1, 0) - 2*phi + np.roll(phi, -1, 0)) +
        (np.roll(phi, 1, 1) - 2*phi + np.roll(phi, -1, 1)) +
        (np.roll(phi, 1, 2) - 2*phi + np.roll(phi, -1, 2))
    ) / (dx**2)

# --- Weak-field Einstein component ---
# In this normalization (∇^2 Phi = rho), the weak-field identity is:
# G00 ≈ 2 ∇^2 Phi  and  8*pi*G*T00 -> 8*pi*T00 with G absorbed -> 8*pi*rho
G00 = 2.0 * laplacian(Phi, dx)
coeff = 8.0 * np.pi
T00 = rho  # static dust in our normalized units

residual_field = G00 - coeff * T00
residual_norm = np.linalg.norm(residual_field) / T00.size

# --- Conservation: static dust, zero flux -> divergence ~ 0 ---
def grad(phi, axis, dx):
    return (np.roll(phi, -1, axis) - np.roll(phi, 1, axis)) / (2.0 * dx)

Ti0 = [np.zeros_like(rho) for _ in range(3)]
div_T0 = grad(Ti0[0], 0, dx) + grad(Ti0[1], 1, dx) + grad(Ti0[2], 2, dx)
divergence_norm = np.linalg.norm(div_T0) / div_T0.size

# --- Diagnostics ---
lap_residual = laplacian(Phi, dx) - rho
lap_res_norm = np.linalg.norm(lap_residual) / rho.size

print(f"Poisson residual norm (FD vs spectral): {lap_res_norm:.3e}")
print(f"Einstein equation residual norm (weak-field G00): {residual_norm:.3e}")
print(f"Stress-energy divergence norm (static dust): {divergence_norm:.3e}")


Poisson residual norm (FD vs spectral): 1.496e-06
Einstein equation residual norm (weak-field G00): 2.103e-04
Stress-energy divergence norm (static dust): 0.000e+00
